In [1]:
import scipy as sp
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, auc
import matplotlib.pyplot as plt
from pathlib import Path  
import scanpy as sc
import os 
from model_evaluation import *
import warnings
warnings.filterwarnings("ignore")

sc.settings.set_figure_params(figsize=(3, 3))

## Utils functions

In [2]:
def compute_precision_and_recall_auc(predicted, true):
    # Compute precision and recall curve and AUC
    precision, recall, _ = precision_recall_curve(true, predicted)
    pr_auc = auc(recall, precision)
    return pr_auc

def cluster_metrics(abs_ratios, ratios, is_abundant, leiden_labels):
    # Non-abundant clusters should always be lower 
    non_abundant_over_abundant = abs_ratios[~is_abundant].mean() / abs_ratios[is_abundant].mean()
    # Compute fraction correct sign
    ratios_abundant = ratios[is_abundant]  # Only clusters 1 and 2 
    leiden_abundant = np.array(leiden_labels)[is_abundant]

    leiden_abundant_unique = np.unique(leiden_abundant)
    assert "0" not in leiden_abundant_unique
    assert "3" not in leiden_abundant_unique
    
    ratios_abundant[leiden_abundant==2] = ratios_abundant[leiden_abundant==2] * (-1)
    correct_sign_prop = (ratios_abundant >= 0).sum() / len(ratios_abundant)
    
    # Kolmogorov-Smirnov test 
    ks_D, _ = sp.stats.ks_2samp(
        abs_ratios[is_abundant],
        abs_ratios[~is_abundant],
        alternative="two-sided",
        mode="asymp"
    )
    w_distance = sp.stats.wasserstein_distance(abs_ratios[is_abundant], abs_ratios[~is_abundant])
    return non_abundant_over_abundant, correct_sign_prop, ks_D, w_distance
    
def compute_evaluation_metrics(result_path, subsampling_rates, loga_ratio_key):
    # Collect resultsper oversampling (only one per run)
    results_per_oversamp = {
        "p_major_cluster": [], 
        "p_minor_cluster": [],
        "auc_score": [], 
        "non_abundant_over_abundant": [],
        "correct_sign_prop": [],
        }
    
    results_per_run = {
        "metric": [], 
        "value": [],
        }

    # Compute the ratios and abundances per cluster
    per_cluster_ratios = []
    per_cluster_log_odds = []
        
    for r_oversamp in subsampling_rates:
        # Compute the true per-cluster ratio 
        p = 0.5 - r_oversamp  # Always minor cluster prop
        one_minus_p = 1 - p  # Always major cluster prop
        csv_generated = pd.read_csv(result_path / f"oversamp_{r_oversamp}.csv")
            
        # Collect abundance labels 
        is_abundant = np.logical_or(csv_generated.leiden==1, csv_generated.leiden==2)
        abs_ratio = np.abs(csv_generated[loga_ratio_key]).values
        ratio = csv_generated[loga_ratio_key].values

        # Compute AUC
        auc = compute_precision_and_recall_auc(abs_ratio, is_abundant)
        results_per_oversamp["p_major_cluster"].append(one_minus_p)
        results_per_oversamp["p_minor_cluster"].append(p)
        results_per_oversamp["auc_score"].append(auc)
            
        for cl in [0, 1, 2, 3]:
            per_cluster_ratios.append(np.mean(ratio[csv_generated["leiden"]==cl]))

            numerator = (csv_generated.loc[csv_generated["leiden"]==cl, "treatment"]==1).sum()
            denominator = (csv_generated.loc[csv_generated["leiden"]==cl, "treatment"]==0).sum() 

            abs_log_odds = np.log((numerator + 1e-9) / (denominator + 1e-9))
            per_cluster_log_odds.append(abs_log_odds)
            
        # Compute cluster metrics
        abundant_over_non_abundant, correct_sign_prop, _, _ = cluster_metrics(abs_ratio, ratio, is_abundant, csv_generated["leiden"])
        results_per_oversamp["non_abundant_over_abundant"].append(abundant_over_non_abundant)
        results_per_oversamp["correct_sign_prop"].append(correct_sign_prop)
        
    results_per_run["metric"].append("spearman_log_odds")
    results_per_run["value"].append(sp.stats.spearmanr(per_cluster_ratios, per_cluster_log_odds)[0])
    results_per_run["metric"].append("pearson_log_odds")
    results_per_run["value"].append(sp.stats.pearsonr(per_cluster_ratios, per_cluster_log_odds)[0])

    # After computing the metrics per run and abundance, compute metrics per run only. 
    results_per_oversamp_df = pd.DataFrame(results_per_oversamp)
    # Take df at run i 
    for metric in results_per_oversamp_df.columns:
        name = f"corr_{metric}_p"
        results_per_run["metric"].append(name)
        results_per_run["value"].append(sp.stats.pearsonr(results_per_oversamp_df[metric],
                                             results_per_oversamp_df["p_major_cluster"])[0])

        name = f"mean_{metric}"
        results_per_oversamp_df_over_03 = results_per_oversamp_df[results_per_oversamp_df.p_major_cluster>0.8]
        results_per_run["metric"].append(name)
        results_per_run["value"].append(results_per_oversamp_df_over_03[metric].mean())
    
    # Results per data frame 
    results_per_run_df = pd.DataFrame(results_per_run)    
    return results_per_oversamp_df, results_per_run_df

Initialize subsampling rates

In [3]:
subsampling_rates = [0.0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.45, 0.50]

Initialize path sweeps 

In [4]:
path_to_sweeps = Path("../../project_folder/results/abundance_test_experiment/scRatio_sweep")

In [5]:
results = {
    "dimensions": [], 
    "batch_size": [],
    "scheduler_type": [], 
    "sigma": [], 
    "solver": []}


for file in os.listdir(path_to_sweeps):
    # Split file 
    file_split = file.split("_")
    if file_split[-1] == "0.25":
        continue
    for solver in ["log_ratios"]:
        results["solver"].append(solver)
        results_per_oversamp_df, results_per_run_df = compute_evaluation_metrics(path_to_sweeps / file,
                                                                                 subsampling_rates, 
                                                                                 solver)
        for key in np.unique(results_per_run_df.metric): 
            if key not in results:
                results[key] = []
            results[key].append(results_per_run_df.loc[results_per_run_df.metric==key]["value"].values[0])
            
    results["dimensions"].append(file_split[1])
    results["batch_size"].append(file_split[4])
    results["scheduler_type"].append(file_split[7])
    results["sigma"].append(file_split[-1])

In [6]:
results_df = pd.DataFrame(results)

In [7]:
results_df.sort_values(by=["mean_auc_score"], ascending=False)

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
39,10,512,stochastic,0.01,log_ratios,0.924367,0.707960,-0.911991,1.0,-1.0,0.959698,0.984014,0.125390,0.95,0.05,0.928238,0.847874
1,10,512,sigmamin,0.01,log_ratios,0.929405,0.651604,-0.944237,1.0,-1.0,0.958440,0.984941,0.129486,0.95,0.05,0.918996,0.868402
0,10,1024,sigmamin,0.01,log_ratios,0.886873,0.689423,-0.905119,1.0,-1.0,0.958237,0.986602,0.134645,0.95,0.05,0.939886,0.879032
60,10,1024,deterministic,0,log_ratios,0.890114,0.676618,-0.893390,1.0,-1.0,0.957407,0.985063,0.128600,0.95,0.05,0.938393,0.880499
41,10,1024,stochastic,0.01,log_ratios,0.910592,0.702678,-0.932403,1.0,-1.0,0.955104,0.984049,0.133461,0.95,0.05,0.932589,0.846408
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27,30,256,stochastic,0.01,log_ratios,0.957147,0.907503,-0.941623,1.0,-1.0,0.810176,0.938521,0.273347,0.95,0.05,0.903929,0.755865
17,50,512,deterministic,0,log_ratios,0.988673,0.954488,-0.989394,1.0,-1.0,0.802192,0.926313,0.282826,0.95,0.05,0.926322,0.806452
30,30,512,stochastic,0.1,log_ratios,0.948105,0.854813,-0.818530,1.0,-1.0,0.788856,0.951586,0.371209,0.95,0.05,-0.549197,0.390029
23,20,256,stochastic,0.1,log_ratios,0.864775,0.840792,-0.920678,1.0,-1.0,0.746779,0.954472,0.329340,0.95,0.05,0.873541,0.739370


In [8]:
results_df.sigma.unique()

array(['0.01', '0.1', '0', '0.5'], dtype=object)

## More in depth analysis 

In [9]:
results_df_sigma_min = results_df[results_df.scheduler_type=="sigmamin"]

In [10]:
results_df_stochastic = results_df[results_df.scheduler_type=="stochastic"]

In [11]:
results_df_deterministic = results_df[results_df.scheduler_type=="deterministic"]

In [12]:
results_df_sigma_min.sort_values(by=["mean_auc_score"], ascending=False)

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
1,10,512,sigmamin,0.01,log_ratios,0.929405,0.651604,-0.944237,1.0,-1.0,0.958440,0.984941,0.129486,0.95,0.05,0.918996,0.868402
0,10,1024,sigmamin,0.01,log_ratios,0.886873,0.689423,-0.905119,1.0,-1.0,0.958237,0.986602,0.134645,0.95,0.05,0.939886,0.879032
40,10,1024,sigmamin,0.1,log_ratios,0.885890,0.717796,-0.906475,1.0,-1.0,0.954881,0.984206,0.130149,0.95,0.05,0.935319,0.880865
31,10,512,sigmamin,0.1,log_ratios,0.922775,0.680337,-0.933288,1.0,-1.0,0.954030,0.983419,0.131826,0.95,0.05,0.921697,0.848974
34,10,256,sigmamin,0.01,log_ratios,0.891175,0.691982,-0.904705,1.0,-1.0,0.941255,0.985203,0.166752,0.95,0.05,0.918535,0.851906
18,10,256,sigmamin,0.1,log_ratios,0.919201,0.745956,-0.918053,1.0,-1.0,0.939782,0.983139,0.161174,0.95,0.05,0.914559,0.856305
58,20,1024,sigmamin,0.01,log_ratios,0.979434,0.861135,-0.958212,1.0,-1.0,0.926931,0.974254,0.159552,0.95,0.05,0.942319,0.850440
19,5,1024,sigmamin,0.1,log_ratios,0.780636,0.600057,-0.800596,1.0,-1.0,0.919113,0.958145,0.135899,0.95,0.05,0.849898,0.835777
4,20,512,sigmamin,0.1,log_ratios,0.968875,0.767099,-0.952951,1.0,-1.0,0.916500,0.969829,0.161848,0.95,0.05,0.923599,0.861804
11,5,256,sigmamin,0.1,log_ratios,0.833768,0.611271,-0.853795,1.0,-1.0,0.913433,0.960664,0.154497,0.95,0.05,0.843730,0.826979


In [13]:
results_df_stochastic.sort_values(by=["mean_auc_score"], ascending=False).head()

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
39,10,512,stochastic,0.01,log_ratios,0.924367,0.707960,-0.911991,1.0,-1.0,0.959698,0.984014,0.125390,0.95,0.05,0.928238,0.847874
41,10,1024,stochastic,0.01,log_ratios,0.910592,0.702678,-0.932403,1.0,-1.0,0.955104,0.984049,0.133461,0.95,0.05,0.932589,0.846408
24,10,1024,stochastic,0.1,log_ratios,0.916628,0.673547,-0.932518,1.0,-1.0,0.952359,0.982737,0.132739,0.95,0.05,0.932858,0.861070
55,10,256,stochastic,0.01,log_ratios,0.892837,0.674512,-0.887597,1.0,-1.0,0.950941,0.985133,0.143585,0.95,0.05,0.908884,0.849340
32,10,512,stochastic,0.1,log_ratios,0.910067,0.717999,-0.886357,1.0,-1.0,0.947446,0.983052,0.140574,0.95,0.05,0.906036,0.858871


In [14]:
results_df_deterministic.sort_values(by=["mean_auc_score"], ascending=False).head()

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
60,10,1024,deterministic,0,log_ratios,0.890114,0.676618,-0.893390,1.0,-1.0,0.957407,0.985063,0.128600,0.95,0.05,0.938393,0.880499
65,10,512,deterministic,0,log_ratios,0.895443,0.709371,-0.889350,1.0,-1.0,0.952124,0.983332,0.139741,0.95,0.05,0.920666,0.875000
26,10,256,deterministic,0,log_ratios,0.932331,0.760128,-0.900879,1.0,-1.0,0.942513,0.983786,0.159902,0.95,0.05,0.906750,0.821848
7,20,512,deterministic,0,log_ratios,0.978781,0.849808,-0.963895,1.0,-1.0,0.917637,0.974202,0.168114,0.95,0.05,0.928143,0.847141
22,5,512,deterministic,0,log_ratios,0.826495,0.592962,-0.788932,1.0,-1.0,0.916636,0.958460,0.151078,0.95,0.05,0.855141,0.843109


In [15]:
results_df_stochastic[results_df_stochastic.sigma=="0.25"]

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds


In [16]:
results_df_stochastic[results_df_stochastic.sigma=="0.5"]

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
8,10,512,stochastic,0.5,log_ratios,0.938986,0.732002,-0.928990,1.0,-1.0,0.934274,0.976895,0.143542,0.95,0.05,0.886334,0.802419
42,20,1024,stochastic,0.5,log_ratios,0.989841,0.864083,-0.316099,1.0,-1.0,0.882227,0.977857,1.208713,0.95,0.05,0.146529,0.641862
54,20,512,stochastic,0.5,log_ratios,0.968162,0.939608,-0.739439,1.0,-1.0,0.886567,0.977560,0.475791,0.95,0.05,0.000533,0.800953
63,10,1024,stochastic,0.5,log_ratios,0.895721,0.718371,-0.895815,1.0,-1.0,0.938338,0.981425,0.154176,0.95,0.05,0.912557,0.836144


In [17]:
results_df_stochastic[results_df_stochastic.dimensions=="50"]

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
51,50,512,stochastic,0.1,log_ratios,0.901098,0.922484,0.486927,1.0,-1.0,0.563333,0.888323,13.450995,0.95,0.05,-0.355131,-0.141129


## Check per batch size - deterministic

In [18]:
results_df_deterministic[results_df_deterministic.batch_size=="256"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
26,10,256,deterministic,0,log_ratios,0.932331,0.760128,-0.900879,1.0,-1.0,0.942513,0.983786,0.159902,0.95,0.05,0.906750,0.821848
43,20,256,deterministic,0,log_ratios,0.969840,0.800203,-0.929413,1.0,-1.0,0.895441,0.968780,0.197514,0.95,0.05,0.928311,0.880499
56,30,256,deterministic,0,log_ratios,0.996676,0.917248,-0.989281,1.0,-1.0,0.882582,0.958005,0.211025,0.95,0.05,0.925723,0.886364
9,5,256,deterministic,0,log_ratios,0.857775,0.625090,-0.856454,1.0,-1.0,0.905383,0.955049,0.165482,0.95,0.05,0.813906,0.835411


In [19]:
results_df_deterministic[results_df_deterministic.batch_size=="512"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
65,10,512,deterministic,0,log_ratios,0.895443,0.709371,-0.889350,1.0,-1.0,0.952124,0.983332,0.139741,0.95,0.05,0.920666,0.875000
7,20,512,deterministic,0,log_ratios,0.978781,0.849808,-0.963895,1.0,-1.0,0.917637,0.974202,0.168114,0.95,0.05,0.928143,0.847141
59,30,512,deterministic,0,log_ratios,0.992146,0.878881,-0.986452,1.0,-1.0,0.891154,0.962326,0.202138,0.95,0.05,0.934001,0.833211
22,5,512,deterministic,0,log_ratios,0.826495,0.592962,-0.788932,1.0,-1.0,0.916636,0.958460,0.151078,0.95,0.05,0.855141,0.843109
17,50,512,deterministic,0,log_ratios,0.988673,0.954488,-0.989394,1.0,-1.0,0.802192,0.926313,0.282826,0.95,0.05,0.926322,0.806452


In [20]:
results_df_deterministic[results_df_deterministic.batch_size=="1024"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
60,10,1024,deterministic,0,log_ratios,0.890114,0.676618,-0.893390,1.0,-1.0,0.957407,0.985063,0.128600,0.95,0.05,0.938393,0.880499
12,20,1024,deterministic,0,log_ratios,0.977338,0.865273,-0.942253,1.0,-1.0,0.908236,0.969881,0.172579,0.95,0.05,0.939457,0.865469
3,30,1024,deterministic,0,log_ratios,0.993300,0.871906,-0.980202,1.0,-1.0,0.863444,0.952216,0.222459,0.95,0.05,0.932006,0.837243
16,5,1024,deterministic,0,log_ratios,0.822560,0.596963,-0.861671,1.0,-1.0,0.913416,0.954980,0.145274,0.95,0.05,0.859164,0.852639


In [21]:
results_df_deterministic[results_df_deterministic.dimensions=="50"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
17,50,512,deterministic,0,log_ratios,0.988673,0.954488,-0.989394,1.0,-1.0,0.802192,0.926313,0.282826,0.95,0.05,0.926322,0.806452


The batch size has effect on: 
* corr auc score - better for 512 
* corr abundant over non abundant better for 20 and for 512
* Everything is perfectly reproducible 

## Check per batch size - stochastic

In [22]:
results_df_stochastic[results_df_stochastic.batch_size=="256"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
44,10,256,stochastic,0.1,log_ratios,0.893371,0.690164,-0.877793,1.0,-1.0,0.930239,0.977087,0.164542,0.95,0.05,0.909477,0.825513
55,10,256,stochastic,0.01,log_ratios,0.892837,0.674512,-0.887597,1.0,-1.0,0.950941,0.985133,0.143585,0.95,0.05,0.908884,0.849340
15,20,256,stochastic,0.01,log_ratios,0.973747,0.793689,-0.946387,1.0,-1.0,0.892586,0.971683,0.198056,0.95,0.05,0.930044,0.807185
23,20,256,stochastic,0.1,log_ratios,0.864775,0.840792,-0.920678,1.0,-1.0,0.746779,0.954472,0.329340,0.95,0.05,0.873541,0.739370
27,30,256,stochastic,0.01,log_ratios,0.957147,0.907503,-0.941623,1.0,-1.0,0.810176,0.938521,0.273347,0.95,0.05,0.903929,0.755865
57,30,256,stochastic,0.1,log_ratios,0.965644,0.880493,-0.942060,1.0,-1.0,0.873991,0.970861,0.234821,0.95,0.05,0.926367,0.759164
37,5,256,stochastic,0.1,log_ratios,0.883474,0.612012,-0.868285,1.0,-1.0,0.905028,0.956501,0.167106,0.95,0.05,0.839596,0.817449
50,5,256,stochastic,0.01,log_ratios,0.879346,0.640820,-0.874314,1.0,-1.0,0.911484,0.963567,0.155964,0.95,0.05,0.859536,0.826613


In [23]:
results_df_stochastic[results_df_stochastic.batch_size=="512"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
8,10,512,stochastic,0.5,log_ratios,0.938986,0.732002,-0.928990,1.0,-1.0,0.934274,0.976895,0.143542,0.95,0.05,0.886334,0.802419
32,10,512,stochastic,0.1,log_ratios,0.910067,0.717999,-0.886357,1.0,-1.0,0.947446,0.983052,0.140574,0.95,0.05,0.906036,0.858871
39,10,512,stochastic,0.01,log_ratios,0.924367,0.707960,-0.911991,1.0,-1.0,0.959698,0.984014,0.125390,0.95,0.05,0.928238,0.847874
2,20,512,stochastic,0.1,log_ratios,0.980375,0.816790,-0.910354,1.0,-1.0,0.872304,0.978837,0.266035,0.95,0.05,0.914174,0.831745
6,20,512,stochastic,0.01,log_ratios,0.975264,0.829067,-0.960431,1.0,-1.0,0.919165,0.976773,0.169354,0.95,0.05,0.937947,0.800587
54,20,512,stochastic,0.5,log_ratios,0.968162,0.939608,-0.739439,1.0,-1.0,0.886567,0.977560,0.475791,0.95,0.05,0.000533,0.800953
30,30,512,stochastic,0.1,log_ratios,0.948105,0.854813,-0.818530,1.0,-1.0,0.788856,0.951586,0.371209,0.95,0.05,-0.549197,0.390029
33,30,512,stochastic,0.01,log_ratios,0.974486,0.854288,-0.969074,1.0,-1.0,0.892758,0.965631,0.208530,0.95,0.05,0.921900,0.787023
25,5,512,stochastic,0.01,log_ratios,0.766543,0.576261,-0.792937,1.0,-1.0,0.915804,0.956624,0.136581,0.95,0.05,0.862555,0.835777
35,5,512,stochastic,0.1,log_ratios,0.821102,0.587368,-0.833647,1.0,-1.0,0.906397,0.950205,0.154628,0.95,0.05,0.815144,0.838343


In [24]:
results_df_stochastic[results_df_stochastic.batch_size=="1024"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
24,10,1024,stochastic,0.1,log_ratios,0.916628,0.673547,-0.932518,1.0,-1.0,0.952359,0.982737,0.132739,0.95,0.05,0.932858,0.861070
41,10,1024,stochastic,0.01,log_ratios,0.910592,0.702678,-0.932403,1.0,-1.0,0.955104,0.984049,0.133461,0.95,0.05,0.932589,0.846408
63,10,1024,stochastic,0.5,log_ratios,0.895721,0.718371,-0.895815,1.0,-1.0,0.938338,0.981425,0.154176,0.95,0.05,0.912557,0.836144
5,20,1024,stochastic,0.01,log_ratios,0.965835,0.834778,-0.954816,1.0,-1.0,0.920153,0.973187,0.175228,0.95,0.05,0.940974,0.864370
42,20,1024,stochastic,0.5,log_ratios,0.989841,0.864083,-0.316099,1.0,-1.0,0.882227,0.977857,1.208713,0.95,0.05,0.146529,0.641862
66,20,1024,stochastic,0.1,log_ratios,0.965547,0.858306,-0.519642,1.0,-1.0,0.910566,0.978697,0.262408,0.95,0.05,0.384730,0.515762
38,30,1024,stochastic,0.01,log_ratios,0.990063,0.894679,-0.962194,1.0,-1.0,0.885781,0.960716,0.215587,0.95,0.05,0.896773,0.758798
49,30,1024,stochastic,0.1,log_ratios,0.975700,0.888007,-0.447621,1.0,-1.0,0.812658,0.949225,1.522440,0.95,0.05,-0.005172,0.230938
45,5,1024,stochastic,0.01,log_ratios,0.790264,0.567558,-0.807713,1.0,-1.0,0.911124,0.951761,0.145708,0.95,0.05,0.857402,0.832478
62,5,1024,stochastic,0.1,log_ratios,0.767385,0.567111,-0.799659,1.0,-1.0,0.913874,0.956484,0.146007,0.95,0.05,0.841680,0.847141


## Check per batch size - sigma min

In [25]:
results_df_sigma_min[results_df_sigma_min.batch_size=="256"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
18,10,256,sigmamin,0.1,log_ratios,0.919201,0.745956,-0.918053,1.0,-1.0,0.939782,0.983139,0.161174,0.95,0.05,0.914559,0.856305
34,10,256,sigmamin,0.01,log_ratios,0.891175,0.691982,-0.904705,1.0,-1.0,0.941255,0.985203,0.166752,0.95,0.05,0.918535,0.851906
13,20,256,sigmamin,0.1,log_ratios,0.993512,0.811049,-0.963834,1.0,-1.0,0.911088,0.972977,0.176120,0.95,0.05,0.932904,0.883065
64,20,256,sigmamin,0.01,log_ratios,0.980861,0.914249,-0.942661,1.0,-1.0,0.901147,0.973100,0.197464,0.95,0.05,0.919772,0.790323
28,30,256,sigmamin,0.1,log_ratios,0.988124,0.921951,-0.992029,1.0,-1.0,0.851910,0.954420,0.240284,0.95,0.05,0.941055,0.799853
47,30,256,sigmamin,0.01,log_ratios,0.988382,0.950586,-0.977841,1.0,-1.0,0.875153,0.962203,0.221684,0.95,0.05,0.915082,0.881232
10,5,256,sigmamin,0.01,log_ratios,0.836542,0.573181,-0.875653,1.0,-1.0,0.907038,0.955994,0.156392,0.95,0.05,0.852833,0.829545
11,5,256,sigmamin,0.1,log_ratios,0.833768,0.611271,-0.853795,1.0,-1.0,0.913433,0.960664,0.154497,0.95,0.05,0.843730,0.826979


In [26]:
results_df_sigma_min[results_df_sigma_min.batch_size=="512"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
1,10,512,sigmamin,0.01,log_ratios,0.929405,0.651604,-0.944237,1.0,-1.0,0.958440,0.984941,0.129486,0.95,0.05,0.918996,0.868402
31,10,512,sigmamin,0.1,log_ratios,0.922775,0.680337,-0.933288,1.0,-1.0,0.954030,0.983419,0.131826,0.95,0.05,0.921697,0.848974
4,20,512,sigmamin,0.1,log_ratios,0.968875,0.767099,-0.952951,1.0,-1.0,0.916500,0.969829,0.161848,0.95,0.05,0.923599,0.861804
61,20,512,sigmamin,0.01,log_ratios,0.969002,0.792202,-0.950074,1.0,-1.0,0.903163,0.975478,0.193970,0.95,0.05,0.934745,0.832111
20,30,512,sigmamin,0.01,log_ratios,0.995152,0.914660,-0.980497,1.0,-1.0,0.878712,0.958460,0.213683,0.95,0.05,0.932822,0.869501
29,30,512,sigmamin,0.1,log_ratios,0.977269,0.918347,-0.962740,1.0,-1.0,0.871484,0.957236,0.226031,0.95,0.05,0.923880,0.864003
36,5,512,sigmamin,0.1,log_ratios,0.770605,0.577832,-0.786294,1.0,-1.0,0.909739,0.954682,0.153306,0.95,0.05,0.881474,0.844208
46,5,512,sigmamin,0.01,log_ratios,0.782111,0.573327,-0.805895,1.0,-1.0,0.911951,0.957691,0.149435,0.95,0.05,0.831190,0.833578
14,50,512,sigmamin,0.1,log_ratios,0.986307,0.966358,-0.988417,1.0,-1.0,0.822099,0.935635,0.264228,0.95,0.05,0.923546,0.819648


In [27]:
results_df_sigma_min[results_df_sigma_min.batch_size=="1024"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
0,10,1024,sigmamin,0.01,log_ratios,0.886873,0.689423,-0.905119,1.0,-1.0,0.958237,0.986602,0.134645,0.95,0.05,0.939886,0.879032
40,10,1024,sigmamin,0.1,log_ratios,0.885890,0.717796,-0.906475,1.0,-1.0,0.954881,0.984206,0.130149,0.95,0.05,0.935319,0.880865
52,20,1024,sigmamin,0.1,log_ratios,0.966588,0.826865,-0.932466,1.0,-1.0,0.894791,0.967957,0.199709,0.95,0.05,0.944399,0.832111
58,20,1024,sigmamin,0.01,log_ratios,0.979434,0.861135,-0.958212,1.0,-1.0,0.926931,0.974254,0.159552,0.95,0.05,0.942319,0.850440
48,30,1024,sigmamin,0.01,log_ratios,0.989119,0.886206,-0.987474,1.0,-1.0,0.876252,0.952968,0.202382,0.95,0.05,0.920661,0.870601
53,30,1024,sigmamin,0.1,log_ratios,0.991301,0.913197,-0.981240,1.0,-1.0,0.897890,0.958110,0.187298,0.95,0.05,0.932588,0.875367
19,5,1024,sigmamin,0.1,log_ratios,0.780636,0.600057,-0.800596,1.0,-1.0,0.919113,0.958145,0.135899,0.95,0.05,0.849898,0.835777
21,5,1024,sigmamin,0.01,log_ratios,0.791173,0.580701,-0.812494,1.0,-1.0,0.912422,0.953545,0.149281,0.95,0.05,0.855973,0.847874
